In [1]:
import pandas as pd

# Veri setini yükleyelim
data = pd.read_csv('data/lip_coordinates.csv')

# Genel bilgileri inceleyelim
print(data.info())
print(data.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1099 entries, 0 to 1098
Data columns (total 81 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   time    1099 non-null   float64
 1   0_x     1099 non-null   int64  
 2   13_x    1099 non-null   int64  
 3   14_x    1099 non-null   int64  
 4   17_x    1099 non-null   int64  
 5   37_x    1099 non-null   int64  
 6   39_x    1099 non-null   int64  
 7   40_x    1099 non-null   int64  
 8   61_x    1099 non-null   int64  
 9   78_x    1099 non-null   int64  
 10  80_x    1099 non-null   int64  
 11  81_x    1099 non-null   int64  
 12  82_x    1099 non-null   int64  
 13  84_x    1099 non-null   int64  
 14  87_x    1099 non-null   int64  
 15  88_x    1099 non-null   int64  
 16  91_x    1099 non-null   int64  
 17  95_x    1099 non-null   int64  
 18  146_x   1099 non-null   int64  
 19  178_x   1099 non-null   int64  
 20  181_x   1099 non-null   int64  
 21  185_x   1099 non-null   int64  
 22  

In [2]:
from statsmodels.tsa.stattools import adfuller

# ADF test fonksiyonu
def adf_test(data):
    results = {}
    for column in data.columns:
        test_result = adfuller(data[column])
        results[column] = {
            'ADF Statistic': test_result[0],
            'p-value': test_result[1],
            'Durağan mı?': test_result[1] < 0.05
        }
    return pd.DataFrame(results).T

# 'time' sütununu çıkararak koordinat verilerini alalım
var_data = data.drop(columns=['time'])

# Durağanlık testi
adf_results = adf_test(var_data)
print(adf_results)

      ADF Statistic   p-value Durağan mı?
0_x       -2.164544  0.219365       False
13_x      -2.203439     0.205       False
14_x      -2.171966  0.216577       False
17_x      -2.247038  0.189618       False
37_x      -2.198147  0.206919       False
...             ...       ...         ...
375_y       -2.6404  0.084932       False
402_y     -2.975668   0.03723        True
405_y      -3.61777   0.00543        True
409_y     -3.527387  0.007311        True
415_y     -2.757832  0.064554       False

[80 rows x 3 columns]


In [3]:
# Veriyi fark alarak durağan hale getirme
var_data_diff = var_data.diff().dropna()

# Tekrar durağanlık testi
adf_results_diff = adf_test(var_data_diff)
print(adf_results_diff)

      ADF Statistic p-value Durağan mı?
0_x      -11.978892     0.0        True
13_x     -10.839581     0.0        True
14_x      -12.06736     0.0        True
17_x     -12.451625     0.0        True
37_x     -11.951368     0.0        True
...             ...     ...         ...
375_y     -9.861467     0.0        True
402_y      -9.76567     0.0        True
405_y      -9.55739     0.0        True
409_y    -10.013571     0.0        True
415_y     -9.754089     0.0        True

[80 rows x 3 columns]


In [5]:
from statsmodels.tsa.api import VAR

# VAR modeli oluştur
model = VAR(var_data_diff)

# Uygun gecikme sayısını belirleme (veri setine göre ayarlanmalı)
lag_selection = model.select_order(maxlags=2)  # maksimum gecikme 1 olarak seçildi
optimal_lag = lag_selection.aic
print("Optimal gecikme:", optimal_lag)

# Modeli eğitme
var_results = model.fit(optimal_lag)

# Sonuçları görüntüle
print(var_results.summary())

C:\Users\Asus\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


Optimal gecikme: 2
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Tue, 11, Mar, 2025
Time:                     21:45:44
--------------------------------------------------------------------
No. of Equations:         80.0000    BIC:                   -90.7703
Nobs:                     1096.00    HQIC:                  -127.292
Log likelihood:          -29594.1    FPE:                1.37262e-65
AIC:                     -149.523    Det(Omega_mle):     2.37450e-70
--------------------------------------------------------------------
Results for equation 0_x
              coefficient       std. error           t-stat            prob
---------------------------------------------------------------------------
const           -0.003080         0.023741           -0.130           0.897
L1.0_x          -0.488352         0.074378           -6.566           0.000
L1.13_x          0.121919         0.074077            1.646    